## צבירה בזמן: cumsum ו-trapezoid

צבירה היא סכום של ערך כפול ‎`dt`‎ לאורך הזמן: מהירות מצטברת למרחק, כוח לשינוי בתנע, הספק לאנרגיה. לצבירה יש שתי צורות:

- **הסך הכולל**, מספר אחד: ‎`(v * dt).sum()`‎.
- **הצבירה עד כל רגע**, סדרה: ‎`np.cumsum(v * dt)`‎. כל איבר הוא סכום האיברים עד אליו, כמו מצבר ששומר את ערכו אחרי כל סיבוב.

‎`cumsum`‎ הפוך ל-‎`diff`‎: הסכום המצטבר של ההפרשים מחזיר את הערכים המקוריים, חוץ מהערך הראשון, ש-‎`diff`‎ מאבד ולכן מוסיפים אותו. בתא הבא מיקומים של כדור מתגלגל, החל מ-0.5 מטר:

```{image} ../../figures/10_2_cumsum_steps.svg
:alt: np.cumsum(d) stacks the steps: each element is the sum of the steps up to it
:align: center
:class: only-light
```

```{image} ../../figures/10_2_cumsum_steps-dark.svg
:alt: np.cumsum(d) stacks the steps: each element is the sum of the steps up to it
:align: center
:class: only-dark
```

In [ ]:
import numpy as np
from scipy import integrate

x = np.array([0.5, 0.75, 1.5, 2.75])    # position (m)
d = np.diff(x)                          # three differences
print(np.cumsum(d))                     # x[1:] - x[0]
print(x[0] + np.cumsum(d))              # x[1:] itself

כשהערכים נמדדים ברגעים עצמם, למשל בחיישן, n מדידות תוחמות n-1 מרווחים. הערך של כל מרווח הוא הממוצע של שתי המדידות שבקצותיו, כמו ‎`t_mid`‎ ב-10.1. ‎`integrate.trapezoid(F, t)`‎ מחשבת בדיוק את הסכום הזה, ולכן היא הכלי לסך כולל מתוך מדידות.

```{image} ../../figures/10_2_trapezoid_slices.svg
:alt: Four force readings give three trapezoids; four slices of reading times dt run one step past the last reading
:align: center
:class: only-light
```

```{image} ../../figures/10_2_trapezoid_slices-dark.svg
:alt: Four force readings give three trapezoids; four slices of reading times dt run one step past the last reading
:align: center
:class: only-dark
```

### דוגמה: מהמהירות חזרה לגובה

אלה המדידות של שלוש הזריקות מ-10.1, והמהירויות מחושבות מהן כמו שם. הצבירה של ‎`vy * dt`‎ לאורך ציר הזמן, ועוד הגובה ההתחלתי, מחזירה את הגבהים; הסכום של ‎`vx * dt`‎ נותן את המרחק האופקי הכולל.

In [ ]:
import numpy as np

# "measured" positions (m) of three throws, from a video at 20 frames/s
rng = np.random.default_rng(6)
v0_true = np.array([19.0, 20.0, 21.0])       # launch speeds (m/s)
angle = np.radians(45)
t = np.round(np.arange(65) * 0.05, 2)        # frame times (s)
x = v0_true[:, None] * np.cos(angle) * t
y = 1.5 + v0_true[:, None] * np.sin(angle) * t - 4.9 * t**2
x = np.round(x + rng.normal(0, 0.003, x.shape), 3)
y = np.round(y + rng.normal(0, 0.003, y.shape), 3)

In [ ]:
dt = t[1] - t[0]
vx = np.diff(x, axis=1) / dt                 # as in 10.1: shape (3, 64)
vy = np.diff(y, axis=1) / dt

y0 = y[:, 0]                                 # the starting heights
y_back = y0[:, None] + np.cumsum(vy * dt, axis=1)   # heights at t[1:]
print(y_back.shape)
print(abs(y_back - y[:, 1:]).max())          # compared with the measurement
print((vx * dt).sum(axis=1))                 # total horizontal distance (m)
print(x[:, -1] - x[:, 0])

ההפרש מהמדידה קטן מ-$10^{-15}$, כלומר אפס עד כדי שגיאות העיגול של המחשב: ‎`cumsum`‎ מבטל את ‎`diff`‎. ‎`y0[:, None]`‎ הוא עמודה, ולכן הגובה ההתחלתי של כל ריצה נוסף לכל השורה שלה. כל מהירות כאן שייכת למרווח, כי היא חושבה מהפרש, ולכן הסכום הפשוט מדויק.

### דוגמה נוספת: דחף בהתנגשות

עגלה במסה 0.5 ק"ג נעה במהירות 0.8 מ'/ש' לעבר חיישן כוח, פוגעת בו וחוזרת. החיישן מדד את הכוח על העגלה כל 5 מילישניות (‎`t_hit`‎ ו-‎`F`‎, בניוטון). הכיוון החיובי הוא הכיוון שאליו העגלה חוזרת, ולכן התנע לפני הפגיעה הוא $p_0 = -0.4$ ק"ג·מ'/ש'.

הדחף, כלומר השינוי בתנע, הוא הצבירה של הכוח לאורך כל ההתנגשות: ‎`trapezoid`‎. התנע בכל רגע הוא $p_0$ ועוד הצבירה עד אותו רגע: ‎`cumsum`‎ על הכוח של כל מרווח.

In [ ]:
# "measured" force (N) on a cart that hits a force sensor, every 5 ms
rng = np.random.default_rng(2)
t_hit = np.round(np.arange(41) * 0.005, 3)
F = np.where((t_hit > 0.05) & (t_hit < 0.15),
             0.7 * np.pi / 0.2 * np.sin(np.pi * (t_hit - 0.05) / 0.1), 0.0)
F = np.round(F + rng.normal(0, 0.03, t_hit.shape), 2)

In [ ]:
m = 0.5                                  # mass of the cart (kg)
p0 = m * (-0.8)                          # momentum before the hit
dt_hit = t_hit[1] - t_hit[0]             # 0.005 s

J = integrate.trapezoid(F, t_hit)        # the impulse (N s)
F_mid = (F[:-1] + F[1:]) / 2             # one force per interval
p = p0 + np.cumsum(F_mid * dt_hit)       # momentum after each interval
print(round(J, 3), round(p[-1] - p0, 3))
print(round(p[-1] / m, 2))               # velocity after the hit (m/s)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].plot(t_hit * 1000, F, "o-", markersize=3)
axs[0].set_xlabel("t (ms)")
axs[0].set_ylabel("F (N)")
axs[0].set_title("Force sensor")
axs[1].plot(t_hit[1:] * 1000, p, "o-", markersize=3, color="tab:green")
axs[1].axhline(0, color="gray", linewidth=0.8)
axs[1].set_xlabel("t (ms)")
axs[1].set_ylabel("p (kg m/s)")
axs[1].set_title("Momentum: p0 plus np.cumsum")
plt.show()

הצבירה עד הרגע האחרון שווה לדחף, כי ‎`trapezoid`‎ מחשבת את אותו סכום של ממוצעים. העגלה חוזרת במהירות של כ-0.6 מ'/ש', פחות מ-0.8: ההתנגשות אינה אלסטית לגמרי.

### באג נפוץ: צעד אחד יותר מדי

n מדידות ברגעים תוחמות n-1 מרווחים. ‎`cumsum`‎ או ‎`.sum()`‎ על כל n הערכים, כפול ‎`dt`‎, סופרים n פרוסות, ולכן מאריכים את זמן המדידה בצעד אחד. הדוגמה הפשוטה ביותר היא עגלה במהירות קבועה של 0.5 מ'/ש', שנמדדת 11 פעמים במשך שנייה אחת: היא עברה 0.5 מטר.

In [ ]:
v_const = 0.5 * np.ones(11)              # constant velocity (m/s)
t_const = np.linspace(0, 1, 11)          # 11 readings in 1 s
dt_const = t_const[1] - t_const[0]
print(np.cumsum(v_const * dt_const))     # wrong: 11 slices
v_mid = (v_const[:-1] + v_const[1:]) / 2   # one value per interval
print(np.cumsum(v_mid * dt_const))       # right: 10 intervals
print(integrate.trapezoid(v_const, t_const))

הסדרה הראשונה מסתיימת ב-0.55 מטר, כאילו העגלה נסעה 1.1 שניות: הערך הראשון שלה, 0.05, הוא כבר המרחק אחרי מרווח אחד. בסדרה הנכונה יש ערך אחד לכל מרווח, והיא שייכת לזמנים ‎`t_const[1:]`‎.

### נסו בעצמכם

מנוע מרים משא. מד הספק מדד את ההספק שהמנוע צורך כל חצי שנייה, במשך 10 שניות (‎`t_motor`‎ ו-‎`P`‎, בוואט). האנרגיה היא הצבירה של ההספק בזמן.

השלימו את הפונקציה ‎`half_energy_time`‎: היא מחשבת את האנרגיה הכוללת עם ‎`trapezoid`‎, את האנרגיה עד סוף כל מרווח עם ‎`cumsum`‎, ומחזירה את הזמן הראשון שבו נמסרה לפחות מחצית האנרגיה הכוללת. בדקו אותה על הדוגמה שבתא, ואחר כך הפעילו אותה על המדידות של המנוע.

In [ ]:
# "measured" power (W) of a motor, every 0.5 s
rng = np.random.default_rng(1)
t_motor = np.round(np.arange(21) * 0.5, 1)
P = np.interp(t_motor, [0, 2, 6, 10], [0, 60, 60, 0])
P = np.round(P + rng.normal(0, 0.5, t_motor.shape), 1)
P[0] = 0.0
P[-1] = 0.0

In [ ]:
fig, ax = plt.subplots()
ax.plot(t_motor, P, "o-")
ax.set_xlabel("t (s)")
ax.set_ylabel("P (W)")
ax.set_title("Measured power of the motor")
plt.show()

In [ ]:
def half_energy_time(P, t):
    """First time (s) by which half of the total energy was delivered."""
    pass   # replace with your code

P_test = np.array([0.0, 10.0, 10.0, 10.0, 0.0])   # power (W)
t_test = np.array([0.0, 1.0, 2.0, 3.0, 4.0])      # time (s)
print(integrate.trapezoid(P_test, t_test))        # total energy (J)
print(half_energy_time(P_test, t_test))
# expected: 30.0, then 2.0

`````{admonition} פתרון
:class: dropdown, tip
```python
def half_energy_time(P, t):
    """First time (s) by which half of the total energy was delivered."""
    E_total = integrate.trapezoid(P, t)
    P_mid = (P[:-1] + P[1:]) / 2                 # one power per interval
    E = np.cumsum(P_mid * np.diff(t))            # energy up to t[1:]
    return t[1:][E >= E_total / 2][0]

print(half_energy_time(P_test, t_test))            # 2.0
print(round(integrate.trapezoid(P, t_motor), 1))   # 420.1
print(half_energy_time(P, t_motor))                # 4.5
```
בדוגמה האנרגיה עד סוף כל מרווח היא 5, 15, 25 ו-30 ג'אול, ומחצית מ-30 נמסרה עד הזמן 2. המנוע צרך כ-420 ג'אול, ומחציתם נמסרה עד 4.5 שניות, לפני אמצע זמן המדידה: בגרף ההספק עולה מהר בהתחלה ויורד לאט בסוף.
`````

### בדקו את עצמכם

שתי השאלות מתייחסות לקוד הבא:

```python
a = np.array([1.0, 4.0, 9.0, 16.0])
print(np.cumsum(a))                  # line A

y = np.array([0.0, 2.0, 4.0])
x = np.array([0.0, 1.0, 2.0])
print(integrate.trapezoid(y, x))     # line B
```

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "מהו המספר השלישי שתדפיס שורה A?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "14.0",
                "correct": true,
                "feedback": "נכון: הסכום המצטבר עד הערך השלישי הוא 1 ועוד 4 ועוד 9."
            },
            {
                "answer": "9.0",
                "correct": false,
                "feedback": "לא: 9.0 הוא הערך השלישי עצמו; cumsum מחבר את כל הערכים עד אליו."
            },
            {
                "answer": "5.0",
                "correct": false,
                "feedback": "לא: 5.0 הוא הסכום המצטבר השני, 1 ועוד 4."
            },
            {
                "answer": "30.0",
                "correct": false,
                "feedback": "לא: 30.0 הוא הסכום של כל ארבעת הערכים, המספר האחרון."
            }
        ]
    },
    {
        "question": "מה תדפיס שורה B?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "4.0",
                "correct": true,
                "feedback": "נכון: השטח מתחת לקו שעולה מ-0 ל-4 לאורך 2 יחידות הוא משולש, 2 כפול 4 חלקי 2."
            },
            {
                "answer": "6.0",
                "correct": false,
                "feedback": "לא: 6 הוא סכום הערכים, לא השטח שמתחתם."
            },
            {
                "answer": "8.0",
                "correct": false,
                "feedback": "לא: 8 הוא שטח של מלבן בגובה 4; הקו מתחיל ב-0."
            },
            {
                "answer": "2.0",
                "correct": false,
                "feedback": "לא: 2 הוא אורך הקטע על ציר x, לא השטח. השטח הוא 1 במרווח הראשון ועוד 3 בשני."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)

### תרגול עצמי

אפליקציה בטלפון מדדה את התאוצה האנכית של מעלית כל 0.1 שניות, אחרי שהחסירה את תאוצת הכובד (‎`t_lift`‎ ו-‎`a_lift`‎). המעלית מתחילה במנוחה. חשבו את מהירות המעלית בכל רגע בעזרת ‎`cumsum`‎, ומהמהירות את הגובה, שוב בעזרת ‎`cumsum`‎. מהי המהירות הגדולה ביותר, ולאיזה גובה עלתה המעלית?

In [ ]:
# "measured" vertical acceleration (m/s^2) of an elevator, every 0.1 s
rng = np.random.default_rng(2)
t_lift = np.round(np.arange(121) * 0.1, 1)
a_lift = np.interp(t_lift, [0, 0.75, 1.25, 2.75, 3.25, 8.75, 9.25, 10.75,
                            11.25, 12], [0, 0, 1, 1, 0, 0, -1, -1, 0, 0])
a_lift = np.round(a_lift + rng.normal(0, 0.02, t_lift.shape), 2)

In [ ]:
fig, ax = plt.subplots()
ax.plot(t_lift, a_lift)
ax.set_xlabel("t (s)")
ax.set_ylabel(r"a (m/s$^2$)")
ax.set_title("Acceleration of the elevator, from a phone")
plt.show()

In [ ]:
# dt_lift = ...
# a_mid = ...
# v_lift = np.concatenate([[0.0], ...])   # at rest at the start
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
dt_lift = t_lift[1] - t_lift[0]
a_mid = (a_lift[:-1] + a_lift[1:]) / 2
v_lift = np.concatenate([[0.0], np.cumsum(a_mid * dt_lift)])
v_mid = (v_lift[:-1] + v_lift[1:]) / 2
z_lift = np.concatenate([[0.0], np.cumsum(v_mid * dt_lift)])
print(round(v_lift.max(), 2), round(z_lift[-1], 1))   # 2.01 16.0
```
‎`np.concatenate`‎ מוסיף בהתחלה את הערך ההתחלתי, 0, ולכן למהירות יש ערך בכל רגע מדידה, כמו לתאוצה, ואפשר לצבור אותה שוב. המהירות הגדולה ביותר היא כ-2 מ'/ש', והמעלית עלתה כ-16 מטר.
`````